# 01 — xBD Dataset EDA
## Phase 1 | Disaster Damage Assessment Minor Project
### Track A: xBD Dataset Exploration

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab, skipping drive mount.')
import os
import json
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from collections import Counter


# Define XBD_ROOT based on config
XBD_ROOT = '/content/drive/MyDrive/Project/disaSense_project/data/raw/xbd/train'

# Define and create output directories
OUTPUT_DIRS = [
    '/content/drive/MyDrive/Project/disaSense_project/data/processed',
    '/content/drive/MyDrive/Project/disaSense_project/data/cache',
    '/content/drive/MyDrive/Project/disaSense_project/results/baselines',
    '/content/drive/MyDrive/Project/disaSense_project/results/confusion_matrices'
]

for d in OUTPUT_DIRS:
    os.makedirs(d, exist_ok=True)

print('Drive mounted and output directories created/verified.')

In [ ]:
def inspect_xbd_structure(xbd_root):
    """
    Inspects the xBD dataset directory structure.
    Counts images, labels, and checks for pairs.
    """
    if not os.path.exists(xbd_root):
        print(f"Error: Upload your xBD dataset folder to Google Drive at: {xbd_root}")
        return None
    
    images_dir = os.path.join(xbd_root, 'images')
    labels_dir = os.path.join(xbd_root, 'labels')
    
    if not os.path.exists(images_dir) or not os.path.exists(labels_dir):
        print(f"Error: Missing images or labels directory in {xbd_root}")
        return None
        
    image_files = os.listdir(images_dir)
    label_files = os.listdir(labels_dir)
    
    pre_images = [f for f in image_files if 'pre_disaster' in f and f.endswith('.png')]
    post_images = [f for f in image_files if 'post_disaster' in f and f.endswith('.png')]
    
    # Extract unique disaster type prefixes
    disaster_prefixes = set([f.split('_')[0] for f in image_files])
    
    pre_image_bases = [f.replace('_pre_disaster.png', '') for f in pre_images]
    post_image_bases = [f.replace('_post_disaster.png', '') for f in post_images]
    label_bases = [f.replace('.json', '') for f in label_files if f.endswith('.json')]
    
    mismatches = 0
    missing_labels = 0
    for base in pre_image_bases:
        if base not in post_image_bases:
            print(f"Warning: Missing post-disaster image for {base}")
            mismatches += 1
        if f"{base}_pre_disaster" not in label_bases:
            print(f"Warning: Missing pre-disaster label for {base}")
            missing_labels += 1
        if f"{base}_post_disaster" not in label_bases:
            print(f"Warning: Missing post-disaster label for {base}")
            missing_labels += 1
            
    summary = {
        'total_pre_images': len(pre_images),
        'total_post_images': len(post_images),
        'total_labels': len(label_files),
        'disaster_types': list(disaster_prefixes),
        'mismatches': mismatches,
        'missing_labels': missing_labels
    }
    
    print("--- xBD Dataset Summary ---")
    for k, v in summary.items():
        print(f"{k}: {v}")
    
    return summary

summary = inspect_xbd_structure(XBD_ROOT)

In [ ]:
def get_disaster_type_counts(xbd_root):
    """
    Counts the number of pre-disaster images per disaster type.
    """
    images_dir = os.path.join(xbd_root, 'images')
    if not os.path.exists(images_dir):
        return pd.DataFrame()
        
    pre_images = [f for f in os.listdir(images_dir) if 'pre_disaster' in f and f.endswith('.png')]
    
    counts = Counter()
    for f in pre_images:
        disaster = f.split('_')[0]
        counts[disaster] += 1
        
    df = pd.DataFrame(counts.items(), columns=['disaster_type', 'num_image_pairs'])
    df = df.sort_values(by='num_image_pairs', ascending=False)
    
    return df

df_disaster = get_disaster_type_counts(XBD_ROOT)

if not df_disaster.empty:
    plt.figure(figsize=(10, 6))
    bars = plt.barh(df_disaster['disaster_type'], df_disaster['num_image_pairs'], color='skyblue')
    plt.title("xBD Dataset — Image Pairs per Disaster Type")
    plt.xlabel("Number of Image Pairs")
    plt.gca().invert_yaxis()
    
    for bar in bars:
        plt.text(bar.get_width(), bar.get_y() + bar.get_height()/2, 
                 f'{int(bar.get_width())}', 
                 va='center', ha='left')
                 
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/Project/disaSense_project/results/baselines/disaster_type_distribution.png')
    plt.show()
    display(df_disaster)

In [ ]:
def get_damage_class_distribution(xbd_root, sample_limit=500):
    """
    Samples post-disaster label JSONs and computes the distribution of damage classes.
    """
    labels_dir = os.path.join(xbd_root, 'labels')
    if not os.path.exists(labels_dir):
        return {}, 0
        
    post_labels = [os.path.join(labels_dir, f) for f in os.listdir(labels_dir) if 'post_disaster.json' in f]
    
    if len(post_labels) > sample_limit:
        post_labels = random.sample(post_labels, sample_limit)
        
    damage_counts = Counter()
    total_buildings = 0
    valid_classes = ["no-damage", "minor-damage", "major-damage", "destroyed"]
    
    for path in post_labels:
        try:
            with open(path, 'r') as f:
                data = json.load(f)
            
            features = data['features']['xy']
            for feat in features:
                subtype = feat['properties'].get('subtype')
                if subtype in valid_classes:
                    damage_counts[subtype] += 1
                    total_buildings += 1
        except Exception as e:
            print(f"Warning: Failed to parse {path} - {e}")
            
    return dict(damage_counts), total_buildings

damage_dist, total_bldgs = get_damage_class_distribution(XBD_ROOT)

if damage_dist:
    classes = list(damage_dist.keys())
    counts = list(damage_dist.values())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Pie chart
    axes[0].pie(counts, labels=classes, autopct='%1.1f%%', startangle=90, colors=sns.color_palette("pastel"))
    axes[0].set_title("Damage Class Proportions")
    
    # Bar chart
    sns.barplot(x=classes, y=counts, ax=axes[1], palette="pastel")
    axes[1].set_title("Damage Class Counts")
    axes[1].set_ylabel("Count")
    
    plt.suptitle(f"xBD Dataset — Damage Class Distribution (sample of {min(500, len(os.listdir(os.path.join(XBD_ROOT, 'labels')) if os.path.exists(XBD_ROOT) else 0))} files)")
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/Project/disaSense_project/results/baselines/damage_class_distribution.png')
    plt.show()
    
    print(f"Total buildings annotated in sample: {total_bldgs}")
    for c, count in damage_dist.items():
        print(f"{c}: {count/total_bldgs*100:.2f}%")

In [ ]:
# Inspect JSON label format
labels_dir = os.path.join(XBD_ROOT, 'labels')
if os.path.exists(labels_dir):
    post_labels = [f for f in os.listdir(labels_dir) if 'post_disaster.json' in f]
    if post_labels:
        example_label = os.path.join(labels_dir, post_labels[0])
        
        with open(example_label, 'r') as f:
            data = json.load(f)
            
        print("--- Full JSON Output (truncated) ---")
        print(json.dumps(data, indent=2)[:1000] + "\n... [truncated] ...\n")
        
        # Parse and display
        features = data['features']['xy']
        num_buildings = len(features)
        unique_classes = set(feat['properties'].get('subtype') for feat in features if feat['properties'].get('subtype'))
        
        print("--- Parsed Information ---")
        print(f"Number of buildings annotated: {num_buildings}")
        print(f"Unique damage classes in this image: {list(unique_classes)}")
        
        if num_buildings > 0:
            first_poly = features[0]['wkt']
            # WKT format is generally POLYGON ((x1 y1, x2 y2, ...))
            coords = first_poly.replace('POLYGON ((', '').replace('))', '').split(', ')[:3]
            print(f"Example building polygon coordinates (first 3): {coords}")
            
        metadata = data['metadata']
        print(f"Image dimensions: {metadata.get('width')} x {metadata.get('height')}")

In [ ]:
def get_bbox_stats(xbd_root, sample_limit=200):
    """
    Computes bounding box statistics from polygon annotations.
    """
    labels_dir = os.path.join(xbd_root, 'labels')
    if not os.path.exists(labels_dir):
        return None
        
    post_labels = [os.path.join(labels_dir, f) for f in os.listdir(labels_dir) if 'post_disaster.json' in f]
    
    if len(post_labels) > sample_limit:
        post_labels = random.sample(post_labels, sample_limit)
        
    widths = []
    heights = []
    
    for path in post_labels:
        try:
            with open(path, 'r') as f:
                data = json.load(f)
            
            for feat in data['features']['xy']:
                wkt = feat['wkt']
                if 'POLYGON' in wkt:
                    coords_str = wkt.replace('POLYGON ((', '').replace('))', '')
                    points = [tuple(map(float, p.split(' '))) for p in coords_str.split(', ')]
                    
                    xs = [p[0] for p in points]
                    ys = [p[1] for p in points]
                    
                    w = max(xs) - min(xs)
                    h = max(ys) - min(ys)
                    
                    widths.append(w)
                    heights.append(h)
        except Exception as e:
            pass
            
    if not widths:
        return None
        
    widths = np.array(widths)
    heights = np.array(heights)
    
    stats = {
        'mean_width': np.mean(widths),
        'std_width': np.std(widths),
        'mean_height': np.mean(heights),
        'std_height': np.std(heights),
        'median_width': np.median(widths),
        'median_height': np.median(heights),
        'pct_under_64px': np.mean((widths < 64) | (heights < 64)) * 100
    }
    
    return stats, widths, heights

result = get_bbox_stats(XBD_ROOT)
if result:
    stats, widths, heights = result
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(widths, bins=50, color='skyblue', edgecolor='black')
    axes[0].axvline(64, color='red', linestyle='dashed', linewidth=2, label='64px min')
    axes[0].set_title("Building Patch Width Distribution")
    axes[0].set_xlabel("Width (pixels)")
    axes[0].set_ylabel("Count")
    axes[0].legend()
    
    axes[1].hist(heights, bins=50, color='lightgreen', edgecolor='black')
    axes[1].axvline(64, color='red', linestyle='dashed', linewidth=2, label='64px min')
    axes[1].set_title("Building Patch Height Distribution")
    axes[1].set_xlabel("Height (pixels)")
    axes[1].set_ylabel("Count")
    axes[1].legend()
    
    plt.suptitle("Building Patch Size Distribution (pre-expansion)")
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/Project/disaSense_project/results/baselines/bbox_stats.png')
    plt.show()
    
    print("--- Bounding Box Statistics ---")
    for k, v in stats.items():
        print(f"{k}: {v:.2f}")

In [ ]:
def visualise_image_pair(xbd_root, image_id, alpha=0.8):
    """
    Visualises pre, post, and post-with-annotations for a given image ID.
    """
    pre_img_path = os.path.join(xbd_root, 'images', f"{image_id}_pre_disaster.png")
    post_img_path = os.path.join(xbd_root, 'images', f"{image_id}_post_disaster.png")
    post_lbl_path = os.path.join(xbd_root, 'labels', f"{image_id}_post_disaster.json")
    
    if not (os.path.exists(pre_img_path) and os.path.exists(post_img_path) and os.path.exists(post_lbl_path)):
        print(f"Warning: Files missing for {image_id}")
        return
        
    pre_img = cv2.imread(pre_img_path)
    post_img = cv2.imread(post_img_path)
    
    if pre_img is None or post_img is None:
        return
        
    pre_img = cv2.cvtColor(pre_img, cv2.COLOR_BGR2RGB)
    post_img = cv2.cvtColor(post_img, cv2.COLOR_BGR2RGB)
    post_annot = post_img.copy()
    
    colors = {
        'no-damage': (0, 255, 0),        # Green
        'minor-damage': (255, 255, 0),   # Yellow
        'major-damage': (255, 165, 0),   # Orange
        'destroyed': (255, 0, 0)         # Red
    }
    
    try:
        with open(post_lbl_path, 'r') as f:
            data = json.load(f)
            
        overlay = post_annot.copy()
        
        for feat in data['features']['xy']:
            wkt = feat['wkt']
            subtype = feat['properties'].get('subtype')
            
            if 'POLYGON' in wkt and subtype in colors:
                coords_str = wkt.replace('POLYGON ((', '').replace('))', '')
                points = [tuple(map(float, p.split(' '))) for p in coords_str.split(', ')]
                pts = np.array(points, np.int32).reshape((-1, 1, 2))
                
                color = colors[subtype]
                cv2.polylines(overlay, [pts], isClosed=True, color=color, thickness=2)
                
        cv2.addWeighted(overlay, alpha, post_annot, 1 - alpha, 0, post_annot)
                
    except Exception as e:
        print(f"Warning: Could not parse polygons for {image_id} - {e}")
        return
        
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(pre_img)
    axes[0].set_title("Pre-disaster")
    axes[0].axis('off')
    
    axes[1].imshow(post_img)
    axes[1].set_title("Post-disaster")
    axes[1].axis('off')
    
    axes[2].imshow(post_annot)
    axes[2].set_title("Post with Damage Annotations")
    axes[2].axis('off')
    
    plt.suptitle(f"{image_id} — Pre | Post | Post with Damage Annotations")
    plt.tight_layout()
    plt.savefig(f'/content/drive/MyDrive/Project/disaSense_project/results/baselines/sample_{image_id}.png')
    plt.show()

# Pick 5 random images, ideally from different disasters
images_dir = os.path.join(XBD_ROOT, 'images')
if os.path.exists(images_dir):
    pre_images = [f for f in os.listdir(images_dir) if 'pre_disaster' in f]
    
    if pre_images:
        # Group by disaster
        disasters = {}
        for f in pre_images:
            d = f.split('_')[0]
            if d not in disasters:
                disasters[d] = []
            disasters[d].append(f.replace('_pre_disaster.png', ''))
            
        sample_ids = []
        for d, ids in disasters.items():
            sample_ids.append(random.choice(ids))
            if len(sample_ids) >= 5:
                break
                
        while len(sample_ids) < 5 and pre_images:
            sample_ids.append(random.choice(pre_images).replace('_pre_disaster.png', ''))
            
        sample_ids = sample_ids[:5]
        
        for idx in sample_ids:
            visualise_image_pair(XBD_ROOT, idx)

In [ ]:
def generate_summary_report():
    """
    Generates and saves a final markdown summary report of EDA.
    """
    if 'summary' not in globals() or not summary:
        print("Run previous cells to generate data.")
        return
        
    report = f"""# xBD Dataset EDA Summary\n    \n## General Statistics\n- Total Image Pairs: {summary.get('total_pre_images')}\n- Total Label Files: {summary.get('total_labels')}\n- Mismatched Pairs: {summary.get('mismatches')}\n- Missing Labels: {summary.get('missing_labels')}\n\n## Disaster Types Found\n{df_disaster.to_string(index=False) if 'df_disaster' in globals() else 'Not available'}\n\n## Damage Class Distribution (Sample)\n"""
    if 'damage_dist' in globals() and 'total_bldgs' in globals():
        for c, count in damage_dist.items():
            report += f"- {c}: {count/total_bldgs*100:.2f}%\n"
            
    report += "\n## Bounding Box Statistics\n"
    if 'stats' in globals():
        report += f"- Median Size: {stats['median_width']:.1f}px x {stats['median_height']:.1f}px\n"
        report += f"- Patches Under 64px: {stats['pct_under_64px']:.1f}%\n"
        
    report += "\n## Recommendations\n"
    if 'df_disaster' in globals() and len(df_disaster) >= 2:
        top1, top2 = df_disaster['disaster_type'].iloc[0], df_disaster['disaster_type'].iloc[1]
        report += f"- Start experiments with: **{top1}** and **{top2}** (highest volume).\n"
        
    report += "\n## Output Paths\n"
    report += "- All figures saved to: `/content/drive/MyDrive/Project/disaSense_project/results/baselines/`\n"
    
    # Save text report
    out_path = '/content/drive/MyDrive/Project/disaSense_project/results/baselines/eda_summary.txt'
    with open(out_path, 'w') as f:
        f.write(report)
        
    from IPython.display import display, Markdown
    display(Markdown(report))

generate_summary_report()